In [12]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import classification_report,accuracy_score

In [2]:
# reading the dataset
df1 = pd.read_csv("Phishing_validation_emails.csv")  
df1['Email Type'] = df1['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})
df1 = df1.drop_duplicates().reset_index(drop=True)

In [3]:
df2 = pd.read_csv("CEAS_08.csv")  
df2 = df2.drop_duplicates().reset_index(drop=True)[["body","label"]]
df2 = df2.rename(columns={"body": "Email Text", "label": "Email Type"})

In [4]:
df = pd.concat([df1, df2], ignore_index=True)

In [5]:
text_col, label_col  = "Email Text", "Email Type"

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df[text_col], df[label_col],
    test_size=0.2, random_state=42, stratify=df[label_col]
)

In [7]:
# feature extraction

class TextStats(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        feats = []
        for text in X:
            if pd.isna(text):
                text = ""
            num_chars = len(text)
            num_words = len(text.split())
            num_exclaims = text.count('!')
            num_questions = text.count('?')
            num_http = len(re.findall(r'http[s]?://', text))
            num_dollars = text.count('$')
            num_digits = sum(c.isdigit() for c in text)
            feats.append([num_chars, num_words, num_exclaims, num_questions,
                          num_http, num_dollars, num_digits])
        return np.array(feats)


class CombinedFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=20000):
        self.tfidf = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            stop_words="english",
            min_df=3
        )
        self.stats = TextStats()

    def fit(self, X, y=None):
        self.tfidf.fit(X)
        self.stats.fit(X)
        return self

    def transform(self, X):
        tfidf_features = self.tfidf.transform(X)
        stats_features = csr_matrix(self.stats.transform(X))
        # horizontally stack sparse and dense features
        return hstack([tfidf_features, stats_features])

    

In [8]:
pipeline = Pipeline([
    ("features", CombinedFeatures()),
    ("clf", LogisticRegression(max_iter=1000))
])

In [9]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

/home/chandaluri/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



✅ Classification Report:


NameError: name 'classification_report' is not defined

In [13]:
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3478
           1       0.99      0.99      0.99      4373

    accuracy                           0.99      7851
   macro avg       0.99      0.99      0.99      7851
weighted avg       0.99      0.99      0.99      7851

Accuracy: 0.9894280983314228


In [14]:
# saving the pipeline for plugin usage
from joblib import dump, load

# Save the pipeline to a file
dump(pipeline, 'phishing_pipeline.pkl')

# testing if the saved pipeline works as expected

# Load the pipeline from the file
loaded_pipeline = load('phishing_pipeline.pkl')

# Make predictions
predictions = loaded_pipeline.predict(X_test)


In [15]:
predictions

array([0, 1, 0, ..., 1, 1, 1])